# 🚀 Autoresearch — [GPU PyTorch] Gemini Edition

Welcome to the **Autoresearch notebook runner**!

This notebook autonomously runs Andrej Karpathy's `autoresearch` framework via a notebook using the Google Gen AI SDK (Gemini).

It establishes an autonomous loop that:
1. Modifies the training script (`train_*.py`)
2. Validates it through a fast iteration
3. Compares the metrics (`val_bpb`, `peak_vram_mb`)
4. Approves or discards the hypothesis

# Imports

In [ ]:
import datetime
import getpass
import json
import os
import re
import subprocess
import sys
import time
import urllib.request

from google import genai

# Core Functions & Helpers
These functions handle logging (`tsv`), extracting Python code from the LLM, committing code, and executing the train pipeline safely.

In [ ]:
def check_cached_data(data_cache_dir):
    if os.path.exists(data_cache_dir) and len(os.listdir(data_cache_dir)) > 2:
        return True
    return False


def read_results_tsv():
    """Reads the results payload. Creates a header if missing."""
    results_path = os.path.join(CURRENT_REPO_DIR, RESULTS_FILE)
    if os.path.exists(results_path):
        with open(results_path) as f:
            return f.read()
    return "commit\tval_bpb\tmemory_gb\tstatus\tdescription\n"


def append_result(
    commit: str, val_bpb: str, memory_gb: str, status: str, description: str
):
    """Writes down the result of an experiment into the active TSV ledger."""
    results_path = os.path.join(CURRENT_REPO_DIR, RESULTS_FILE)
    if not os.path.exists(results_path):
        with open(results_path, "w") as f:
            f.write("commit\tval_bpb\tmemory_gb\tstatus\tdescription\n")
    with open(results_path, "a") as f:
        f.write(f"{commit}\t{val_bpb}\t{memory_gb}\t{status}\t{description}\n")


def get_short_commit() -> str:
    """Retrieves the current git short commit hash."""
    r = subprocess.run(
        ["git", "rev-parse", "--short", "HEAD"],
        capture_output=True,
        text=True,
        cwd=CURRENT_REPO_DIR,
    )
    return r.stdout.strip()


def run_training():
    """
    Executes the training script using uv or system python. Returns metrics and raw logs.
    """
    cmd = [sys.executable, TRAIN_FILE] if "tpu_pytorch" in TRAIN_FILE else ["uv", "run", "--no-sync", TRAIN_FILE]
    print("cmd is:", cmd)
    try:
        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            timeout=700,
            cwd=CURRENT_REPO_DIR,
        )
        output = result.stdout + result.stderr

        # Keep physical log file
        with open(os.path.join(CURRENT_REPO_DIR, "run.log"), "w") as f:
            f.write(output)

        if result.returncode != 0:
            return None, None, output

        val_bpb = None
        peak_vram = None
        for line in output.split("\n"):
            if line.startswith("val_bpb:"):
                val_bpb = float(line.split()[-1])
            elif line.startswith("peak_vram_mb:"):
                peak_vram = float(line.split()[-1])

        return val_bpb, peak_vram, output

    except subprocess.TimeoutExpired:
        return None, None, "TIMEOUT: Training exceeded 700s limit"
    except Exception as e:
        return None, None, f"ERROR: {str(e)}"


def extract_code_block(text: str) -> str | None:
    """
    Intelligently extracts Python code blocks from the Gemini markdown response.
    """
    pattern_py = r"```python\s*\n(.*?)```"
    matches = re.findall(pattern_py, text, re.DOTALL)
    if matches:
        return max(matches, key=len)

    pattern_fallback = r"```\s*\n(.*?)```"
    matches_fallback = re.findall(pattern_fallback, text, re.DOTALL)
    if matches_fallback:
        return max(matches_fallback, key=len)

    return None


def push_results_to_branch(branch_name: str):
    """Commits and pushes the results to the remote branch."""
    print(f"\n📤 Pushing results and code to remote branch '{branch_name}'...")
    subprocess.run(
        ["git", "add", RESULTS_FILE, TRAIN_FILE],
        cwd=CURRENT_REPO_DIR,
        capture_output=True,
    )
    subprocess.run(
        ["git", "commit", "-m", "chore: update autoresearch results"],
        cwd=CURRENT_REPO_DIR,
        capture_output=True,
    )
    return subprocess.run(
        ["git", "push", "-u", "origin", "HEAD"],
        cwd=CURRENT_REPO_DIR,
        capture_output=True,
        text=True,
    )


def create_pull_request(branch_name: str, tag: str, repo_url: str, target_branch: str):
    """Creates a pull request using the GitHub API."""
    print("✅ Successfully pushed to remote! Creating PR...")
    cmd = [sys.executable, TRAIN_FILE] if "tpu" in TRAIN_FILE else ["uv", "run", "--no-sync", TRAIN_FILE]
    try:
        repo_path = repo_url.split("github.com/")[1].replace(".git", "")
        api_url = f"https://api.github.com/repos/{repo_path}/pulls"
        headers = {
            "Authorization": f"token {os.environ.get('GITHUB_TOKEN')}",
            "Accept": "application/vnd.github.v3+json",
        }
        data = {
            "title": f"🚀 Autoresearch update: {tag}",
            "head": branch_name,
            "base": target_branch,
            "body": "Automatically generated PR from autoresearch notebook runner.",
        }
        req = urllib.request.Request(
            api_url, headers=headers, data=json.dumps(data).encode("utf-8")
        )
        with urllib.request.urlopen(req) as res:
            body = json.loads(res.read())
            print(f"✅ Pull Request created successfully: {body.get('html_url')}")
    except Exception as e:
        err_msg = getattr(e, "read", lambda: b"")().decode("utf-8", errors="ignore")
        print(f"❌ Failed to create PR: {e} {err_msg}")

# Parameters

In [ ]:
REPO_URL = "https://github.com/dimitreOliveira/autoresearch.git"
REPO_DIR = "autoresearch"
RESULTS_FILE = "results.tsv"
TRAIN_FILE = (
    "train_tpu_pytorch.py"  # One of (train_gpu_pytorch.py, train_tpu_pytorch.py)
)
MODEL_ID = "gemini-3.1-pro-preview"
MAX_EXPERIMENTS = 5  # 10

# Switch to isolated experimental branch mapped to today
TAG = datetime.datetime.now().strftime("%b%d%H").lower()
BRANCH = f"{REPO_DIR}/{TAG}"
TARGET_BRANCH = "tpu_pytorch_support"  # "master" # Replace if you want to run or target another branch

# Current working directory is set by %cd previously
CURRENT_REPO_DIR = os.path.abspath(REPO_DIR)
data_cache_dir = os.path.expanduser(f"~/.cache/{REPO_DIR}/data")

gemini_api_key = getpass.getpass("Please enter your Gemini API Key manually: ")
github_token = getpass.getpass("Please enter your GitHub Token manually: ")

# Export the token as an environment variable for secure git interactions
os.environ["GITHUB_TOKEN"] = github_token

# Repository Configuration

Run script to clone the repository and configure git if missing

In [ ]:
print("⏳ Preparing notebook environment, cloning and configuring git...")

# Clone the repository if missing
if not os.path.exists(REPO_DIR):
    print("Cloning repository...")
    subprocess.run(
        ["git", "clone", "--branch", TARGET_BRANCH, REPO_URL, REPO_DIR],
        capture_output=True,
    )
else:
    print(f"✅ Repository already exists at {REPO_DIR}")

!uv run {CURRENT_REPO_DIR}/prepare_notebook.py --repo-dir {CURRENT_REPO_DIR}

# Data Preparation (Tokenizer & Shards)
Download the necessary data splits and train the GPT tokenizer. This operation is cached locally after the first run.

In [ ]:
if check_cached_data(data_cache_dir):
    print("✅ Data already cached. Skipping download.")
else:
    print("⏳ Downloading data shards and training tokenizer (one-time, ~2-3 min)...")
    !uv run {CURRENT_REPO_DIR}/prepare.py --num-shards 4

# Configure Gemini Platform SDK

In [ ]:
client = genai.Client(api_key=gemini_api_key)

# Execute Autonomous Pipeline
Launch the Autoresearch pipeline. The LLM acts autonomously against the local repository by analyzing feedback, making code iterations, validating outputs, and optimizing toward a lower `val_bpb`.

In [ ]:
# Prep context state
subprocess.run(["git", "add", "-A"], cwd=CURRENT_REPO_DIR)
subprocess.run(["git", "diff", "--cached", "--quiet"], cwd=CURRENT_REPO_DIR)

r = subprocess.run(
    ["git", "log", "--oneline", "-1"],
    capture_output=True,
    text=True,
    cwd=CURRENT_REPO_DIR,
)
if r.returncode != 0:
    subprocess.run(["git", "commit", "-m", "initial"], cwd=CURRENT_REPO_DIR)

subprocess.run(
    ["git", "checkout", "-b", BRANCH], cwd=CURRENT_REPO_DIR, capture_output=True
)

with open(os.path.join(CURRENT_REPO_DIR, "program.md")) as f:
    SYSTEM_PROMPT = f.read()

with open(os.path.join(CURRENT_REPO_DIR, TRAIN_FILE)) as f:
    current_code = f.read()

In [ ]:
# Build baseline chat execution state
history = []
results_context = read_results_tsv()
user_msg = (
    f"Here is the name of the current training file {TRAIN_FILE}:\n\n```python\n{current_code}\n```\n\n"
    f"Run the baseline first (no changes). Then start experimenting.\n"
    f"For each experiment, respond with:\n"
    f"1. A short description of what you are trying (1 sentence)\n"
    f"2. The COMPLETE modified {TRAIN_FILE} in a ```python code block```\n\n"
    f"If this is the baseline run, just say 'BASELINE' and return the code unchanged."
)
history.append({"role": "user", "parts": [{"text": user_msg}]})

best_bpb = None
print(f"🚀 Starting autonomous research loop (max {MAX_EXPERIMENTS} iterations)")

# Main Training Pipeline Loop
for exp_num in range(MAX_EXPERIMENTS):
    print(f"\n{'=' * 70}")
    print(f"🪐 EXPERIMENT #{exp_num}")
    print(f"{'=' * 70}")
    print("🤖 Gemini is thinking...")

    try:
        response = client.models.generate_content(
            model=MODEL_ID,
            contents=history,
            config=genai.types.GenerateContentConfig(
                system_instruction=SYSTEM_PROMPT,
                temperature=0.7,
            ),
        )
        response_text = response.text
    except Exception as e:
        print(f"⚠️ LLM Request Failed: {e}")
        time.sleep(30)
        continue

    # Identify objective descriptions
    description = "unknown experiment"
    for line in response_text.split("\n"):
        line = line.strip()
        if (
            line
            and not line.startswith("```")
            and not line.startswith("import")
            and len(line) > 5
        ):
            description = line.lstrip("#").lstrip("*").strip()[:100]
            break

    # Determine validation baseline run status
    is_baseline = "baseline" in response_text.lower()[:200] and exp_num == 0
    if is_baseline:
        description = "baseline"

    print(f"📝 Objective: {description}")

    # Write generated python segment into the train file execution block
    new_code = extract_code_block(response_text)
    if new_code and len(new_code) > 100:
        with open(os.path.join(CURRENT_REPO_DIR, TRAIN_FILE), "w") as f:
            f.write(new_code)
    elif not is_baseline:
        print(
            "⚠️ WARNING: Could not explicitly infer python block from response. Continuing with active code state."
        )
        description = f"(code extraction failed) {description}"

    # Commit execution log snapshot
    subprocess.run(["git", "add", TRAIN_FILE], cwd=CURRENT_REPO_DIR)
    subprocess.run(
        ["git", "commit", "--allow-empty", "-m", f"exp{exp_num}: {description[:60]}"],
        cwd=CURRENT_REPO_DIR,
        capture_output=True,
    )
    commit = get_short_commit()

    print("🏃‍♂️ Benchmarking validation loop (~5 mins)...")
    t_start = time.time()
    val_bpb, peak_vram, output = run_training()
    t_elapsed = time.time() - t_start
    print(f"⌚️ Training epoch completed sequentially in {t_elapsed:.0f}s")

    # Evaluate execution crash vs stable validations
    if val_bpb is None:
        print("💥 CRASH DETECTED!")
        last_lines = "\n".join(output.split("\n")[-20:])
        print(f"Latest Error Stacktrace:\n{last_lines}")
        append_result(commit, "0.000000", "0.0", "crash", description)

        # Hard reset onto the successful path state
        subprocess.run(
            ["git", "reset", "--hard", "HEAD~1"],
            cwd=CURRENT_REPO_DIR,
            capture_output=True,
        )
        with open(os.path.join(CURRENT_REPO_DIR, TRAIN_FILE)) as f:
            current_code = f.read()

        feedback = (
            f"Experiment #{exp_num} CRASHED. Error:\n```\n{last_lines}\n```\n\n"
            f"Reverted to previous working state.\n"
            f"Please revise your code logic and try a modified approach. Respond with description + complete {TRAIN_FILE}."
        )
    else:
        memory_gb = peak_vram / 1024 if peak_vram else 0
        print(f"📊 metrics — val_bpb: {val_bpb:.6f} | peak_vram: {memory_gb:.1f} GB")

        # Compare model bounds vs historic baseline constraints
        if best_bpb is None or val_bpb < best_bpb:
            status = "keep"
            improvement = 0 if best_bpb is None else best_bpb - val_bpb
            best_bpb = val_bpb
            print(f"⭐ POSITIVE GRADIENT: KEEP! (Improvement Delta: {improvement:.6f})")
        else:
            status = "discard"
            print(f"🚫 REGRESSION: DISCARD (Best metric maintained as {best_bpb:.6f})")
            subprocess.run(
                ["git", "reset", "--hard", "HEAD~1"],
                cwd=CURRENT_REPO_DIR,
                capture_output=True,
            )

        with open(os.path.join(CURRENT_REPO_DIR, TRAIN_FILE)) as f:
            current_code = f.read()

        append_result(commit, f"{val_bpb:.6f}", f"{memory_gb:.1f}", status, description)

        feedback = (
            f"Experiment #{exp_num} result: val_bpb={val_bpb:.6f}, peak_vram={memory_gb:.1f}GB, status={status}\n"
            f"Best val_bpb threshold baseline metric limit: {best_bpb:.6f}\n\n"
            f"Full tracking results history:\n```\n{read_results_tsv()}```\n\n"
            f"Current active code state:\n```python\n{current_code}\n```\n\n"
            f"Generate a robust new hypothesis that alters training loop structures or model configuration positively. Respond with a concise action statement + explicit ```python block."
        )

    # Re-feed cyclic iteration metadata context stream to Gemini
    history.append({"role": "model", "parts": [{"text": response_text}]})
    history.append({"role": "user", "parts": [{"text": feedback}]})

    # Compress LLM history pipeline window buffer to conserve context thresholds footprint bounds
    if len(history) > 10:
        history = [history[0]] + history[-8:]

print("\n" + "=" * 70)
print(
    f"✅ DONE! Terminated completion condition after {MAX_EXPERIMENTS} standard autonomous cyclic test runs."
)
if best_bpb is not None:
    print(f"🏆 Ultimate Peak Performance: val_bpb={best_bpb:.6f}")

push_res = push_results_to_branch(BRANCH)

if push_res.returncode == 0:
    create_pull_request(BRANCH, TAG, REPO_URL, TARGET_BRANCH)
else:
    print(f"❌ Failed to push branch:\n{push_res.stderr}")